# AnyNet Design Space
- Nhìn chung các kiến trúc từ trước tới giờ đều dựa trên trực giác, chưa khám phá không gian thiết kế, không biết tại sao design này tốt, còn design nào tốt hơn không
- Tìm cách kết nối các block chứa các phép conv thông minh để rút ra design pattern tức là tìm ra quy luật thiết kế phù hợp -> AnyNet
- AnyNet gồm 3 phần chính Stem, Body và Head
 + Stem nhận ảnh đầu vào, giảm kích thước ảnh xuống 1 phần còn c_i
 + Body là khối quan trong nhất. Body chứa nhiều stage (module), mỗi stage lại có nhiều block. Trong block lần lượt là các lớp conv 1x1 -> 3x3 -> 1x1 có tác dụng là giảm số lượng kênh đầu vào đi k_i lần (bottleneck), học feature ảnh qua g_i group, tức là mỗi group có c_i/k_i/g_i channel, sau khi học xong, concat g_i group lại về kích thước c_i/k_i channel rồi đi qua 1x1 conv cuối để khôi phục lại kích thước channel ban đầu c_i
 + Cần lưu ý mỗi block đầu của mỗi stage cần giảm độ phân giải ảnh đi 1 nửa và tăng số lượng kênh lên gấp đôi bằng các dùng skip connection 1x1 conv với stride = 2 và số lượng kênh gấp đôi

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
class ResNeXtBlock(nn.Module):
    """The ResNeXt block."""
    def __init__(self, num_channels, groups, bot_mul, use_1x1conv=False, # bot_mul < 1 giảm số kệnh trước khi chia thành g nhóm
                 strides=1):
        super().__init__()
        bot_channels = int(round(num_channels * bot_mul)) # tổng số channel sau khi giảm số lượng
        self.conv1 = nn.LazyConv2d(bot_channels, kernel_size=1, stride=1)
        self.conv2 = nn.LazyConv2d(bot_channels, kernel_size=3,
                                   stride=strides, padding=1,
                                   groups=bot_channels//groups) # groups parameter ở đây là số lượng channel của 1 group (in_channel) ko phải số lượng group
        self.conv3 = nn.LazyConv2d(num_channels, kernel_size=1, stride=1)
        self.bn1 = nn.LazyBatchNorm2d()
        self.bn2 = nn.LazyBatchNorm2d()
        self.bn3 = nn.LazyBatchNorm2d()
        if use_1x1conv:
            self.conv4 = nn.LazyConv2d(num_channels, kernel_size=1,
                                       stride=strides)
            self.bn4 = nn.LazyBatchNorm2d()
        else:
            self.conv4 = None

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = F.relu(self.bn2(self.conv2(Y)))
        Y = self.bn3(self.conv3(Y))
        if self.conv4:
            X = self.bn4(self.conv4(X))
        return F.relu(Y + X)

In [3]:
class AnyNet(nn.Module):
    def stem(self, num_channels):
        return nn.Sequential(
            nn.LazyConv2d(num_channels, kernel_size=3, stride=2, padding=1),
            nn.LazyBatchNorm2d(), 
            nn.ReLU())
    
    def stage(self, depth, num_channels, groups, bot_mul):
        blk = []
        for i in range(depth):
            if i == 0:
                blk.append(ResNeXtBlock(num_channels, groups, bot_mul,
                    use_1x1conv=True, strides=2))
            else:
                blk.append(ResNeXtBlock(num_channels, groups, bot_mul))
        return nn.Sequential(*blk)
    
    def head(self, num_class):
        return nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.LazyLinear(num_class)
        )

    def __init__(self, arch, stem_channels, lr=0.1, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(self.stem(stem_channels))
        for i, s in enumerate(arch):
            self.net.add_module(f'stage{i+1}', self.stage(*s))
        self.net.add_module('head', self.head(num_classes))
        

In [4]:
class RegNetX32(AnyNet):
    def __init__(self, num_classes=10):
        stem_channels, groups, bot_mul = 32, 16, 1
        depths, channels = (4, 6), (32, 80)
        super().__init__(
            ((depths[0], channels[0], groups, bot_mul),
             (depths[1], channels[1], groups, bot_mul)),
            stem_channels, num_classes)
        
    def layer_summary(self, X_shape):
        X = torch.randn(X_shape)
        for layer in self.net:
            X = layer(X)
            print(layer.__class__.__name__, 'output shape:\t', X.shape)

In [5]:
RegNetX32().layer_summary((1, 1, 96, 96))

Sequential output shape:	 torch.Size([1, 32, 48, 48])
Sequential output shape:	 torch.Size([1, 32, 24, 24])
Sequential output shape:	 torch.Size([1, 80, 12, 12])
Sequential output shape:	 torch.Size([1, 10])
